In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime,timedelta
import pandas as pd
import requests
import flexpolyline  # HERE Flexible Polyline 解码
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from math import radians, sin, cos, asin, sqrt
import json
import os 
import sys
PROJECT_ROOT = os.path.abspath("../..")
sys.path.append(PROJECT_ROOT)
api_keys = json.load(open('../../src/api/api_keys.json'))

In [ ]:
color_dict = {
    'measurement': 'red',
    'simulation': '#1808F7',
    'simulation+pred': '#03A719'
}

In [ ]:
# 定义需要批量实验的参数
# 车辆注册号、开始日期、结束日期
vehicle_registration = 'AY71UCD'  # 车辆注册号
start_date, end_date= '20250226','20250430' # 开始日期和结束日期

In [ ]:
# 生成待测试文件名列表
file_name_ls = []
for file_name in os.listdir(os.path.join(PROJECT_ROOT, 'data', 'processed', vehicle_registration)):
    # 提取文件名中的日期部分并判断是否在范围内
    if file_name.endswith('.csv'):
        date_part = file_name[:8]
        if start_date <= date_part <= end_date:
            file_name_ls.append(file_name)
file_name_ls

In [ ]:
# 生成via点和passThrough
def generate_via_points_and_pass_through(df, num_points=30):
    '''
    取df的num_points个点作为via点，起点和终点不包括在内
    :param df: 输入的DataFrame，包含经纬度信息
    :param num_points: 需要生成的via点数量
    :return: via_points: 生成的via点列表，pass_through: 是否经过这些via点的布尔列表
    '''
    if not (2 <= num_points <= min(100, len(df) - 2)):
        raise ValueError("num_points must be between 2 and min(100, len(df)-2).")
    via_points = []
    total = len(df)
    # 起点和终点不包括在内，所以i从1到num_points-1
    for i in range(1, num_points):
        idx = int(i * total / (num_points + 1))
        if idx > 0 and idx < total - 1:
            via_points.append((df.iloc[idx].Latitude, df.iloc[idx].Longitude))
    pass_through = [True] * len(via_points)
    return via_points, pass_through

In [ ]:
import folium
def plot_trajectory_on_map(raw_df, route_df, result_save_dir, GoogleAPIClientInstance):
    """
    在folium地图上绘制测量轨迹和路由轨迹，并标注起点和终点，保存为html文件。
    """
    m = folium.Map(location=[raw_df["Latitude"].mean(), raw_df["Longitude"].mean()], zoom_start=9)
    # 添加Google地图瓦片
    folium.TileLayer(
        tiles=GoogleAPIClientInstance.get_map_tiles(),
        attr='Google',
        name='Google Maps',
        overlay=True,
        control=True,
        max_zoom=20,
        min_zoom=1
    ).add_to(m)
    # 绘制测量数据轨迹
    folium.PolyLine(
        locations=raw_df[["Latitude", "Longitude"]].values.tolist(),
        color="red",
        weight=5,
        popup="Measured Data (GPS)"
    ).add_to(m)
    # 绘制路由数据轨迹
    folium.PolyLine(
        locations=route_df[["Lat", "Lon"]].values.tolist(),
        color="blue",
        weight=3,
        popup="routing Data (HERE API)"
    ).add_to(m)
    # 标注起点
    start_lat, start_lon = raw_df.iloc[0]["Latitude"], raw_df.iloc[0]["Longitude"]
    folium.Marker(
        location=[start_lat, start_lon],
        popup="DEPART",
        icon=folium.Icon(color="green", icon="play")
    ).add_to(m)
    # 标注终点
    end_lat, end_lon = raw_df.iloc[-1]["Latitude"], raw_df.iloc[-1]["Longitude"]
    folium.Marker(
        location=[end_lat, end_lon],
        popup="ARRIVE",
        icon=folium.Icon(color="red", icon="stop")
    ).add_to(m)
    m.add_child(folium.LatLngPopup())
    # 保存地图到文件
    map_file_path = os.path.join(result_save_dir, "measurement&route_on_map.html")
    m.save(map_file_path)
    print(f"Saving map to {map_file_path}")
    return m

In [ ]:
# 导入自定义模块
%load_ext autoreload
%autoreload 2 
from src.api.here_api import HereAPIClient
from src.api.google_api import GoogleAPIClient
from src.api.srf_api import SRFAPIClient
from src.dcgen.dcgen_v3 import driving_cycle_generator
import src.ecp_models.lvd as lvd
HereAPIClientInstance = HereAPIClient(api_keys.get('here_api_key')) # 初始化 HERE API
GoogleAPIClientInstance = GoogleAPIClient(api_keys.get('google_api_key')) # 初始化
SRFAPIClientInsance = SRFAPIClient(api_keys['srf_data_TOKEN'])

In [ ]:
# 车辆参数
crr = 0.0064 # rolling resistance coefficient
cd = 0.46

# 定义物理模型关键参数，这辆车根据实际情况调整了
rolling_resistance_coeff = crr
drag_coefficient = cd
frontal_area_m2 = 10.0
heating_value_mj_l=36.0  # 燃料的低位热值 (MJ/L)
engine_efficiency=0.42  # 发动机效率，默认为0.42
powertrain_efficiency=0.95  # 传动效率

for file_name in file_name_ls:
    file_path_test = os.path.join(PROJECT_ROOT, "data", "processed", vehicle_registration, file_name)
    
    # 读取输入数据
    raw_df = pd.read_csv(file_path_test)
    # print(raw_df.columns)
    origin = raw_df.iloc[0].Latitude, raw_df.iloc[0].Longitude
    destination = raw_df.iloc[-1].Latitude, raw_df.iloc[-1].Longitude
    raw_df["timestamp"] = pd.to_datetime(raw_df["UnixTime"], unit='ms')
    departure_time = raw_df.iloc[0].timestamp
    print(f"Loading file: {file_name}")
    print("duration:", pd.to_datetime(raw_df.iloc[-1].UnixTime, unit='ms') - pd.to_datetime(raw_df.iloc[0].UnixTime, unit='ms')) # 转化为时分秒
    print("departure_time:", departure_time)
    print("distance_measured:", raw_df.iloc[-1].Distance - raw_df.iloc[0].Distance)
    print("distance_gps:", raw_df.iloc[-1].distance_gps - raw_df.iloc[0].distance_gps)
    try:
        avg_wind_speed = raw_df['wind_speed_mps'].mean()
    except Exception as e:
        print("Error calculating avg_wind_speed:", e)   
        avg_wind_speed = 999
    # 定义结果保存路径
    result_save_dir = os.path.join('results', file_name.split('.')[0])
    os.makedirs(result_save_dir, exist_ok=True)
    request_route_path = os.path.join(result_save_dir, 'route.csv')  # 定义请求的routing的结果的路径
    predicted_cycle_path = os.path.join(result_save_dir, 'predicted_cycle.csv') # 定义预测的drving cycle的结果的路径

    # 生成via和passThrough
    via_points, pass_through = generate_via_points_and_pass_through(raw_df, num_points=30)

    if os.path.exists(request_route_path): # 如果请求的routing结果已经存在，则直接读取
        route_df = pd.read_csv(request_route_path)
    else: # 获取路线响应
        resp = HereAPIClientInstance.get_route_resp(
            origin=origin,
            destination=destination,
            via=via_points,
            passThrough=pass_through,
            departure_time=departure_time,
            transportMode="Truck"
        )
        # 将响应转换为 DataFrame
        route_df = HereAPIClientInstance.resp2df(resp)
        route_df.to_csv(os.path.join(result_save_dir, 'route.csv'), index=False)
        print("Route response saved to:", os.path.join(result_save_dir, 'route.csv'))
    
    # 画出测量轨迹和路由轨迹并保存
    m = plot_trajectory_on_map(raw_df, route_df, result_save_dir, GoogleAPIClientInstance)

    # 生成驾驶循环
    if not os.path.exists(predicted_cycle_path):
        print(f"Generating predicted driving cycle and saving to {predicted_cycle_path}...")
        
        dcgen = driving_cycle_generator()
        # 创建驾驶工况
        dc_df = dcgen.create_driving_cycle(
            route_df=route_df,
            start_time=departure_time,
            v_cap=25,  # m/s, 25 m/s = 90 km/h, 卡车的最大速度
            v_roundaboutEnter=3.0,  # m/s
            v_turn=4.0,  # m/s
            a_acc=0.58,  # 加速度的默认参考值，m/s² 
            a_dec=0.83,  # 减速度的默认参考值，m/s²
            dt=1.0,  # 仿真时间步长，单位：秒
            v_cruise=24.1,  # m/s, 24.1 m/s = 86.76 km/h, 卡车的巡航速度
            smooth_speed=True # 是否对速度曲线进行平滑处理
        )
        dc_df = SRFAPIClientInsance.get_elevation_from_srf_database(dc_df)
        dc_df.to_csv(predicted_cycle_path, index=False)
        print(f"Predicted driving cycle saved to {predicted_cycle_path}")
    else:
        print(f"Loading predicted driving cycle from {predicted_cycle_path}")
        dc_df = pd.read_csv(predicted_cycle_path)
        dc_df['timestamp'] = pd.to_datetime(dc_df['timestamp'])

    # debug：打印路由数据的唯一Action类型
    print(route_df.Action.unique())
    

    # 计算平均车辆质量，用于功率和燃油消耗率的计算
    v_mass = raw_df['MassKg'].dropna().mean()
    print("average mass:", v_mass)

    # 计算pred. cycle的功率和燃油消耗率
    dc_df['p_wheel'] = dc_df.apply(lambda row: lvd.calculate_wheel_power(
        mass_kg=v_mass,
        gradient_degrees=row['road_gradient'],
        velocity_mps=row['speed'],
        acceleration_mps2=row['acc'],
        rolling_resistance_coeff = rolling_resistance_coeff, # !!!这辆车的滚动阻力系数根据实际情况调整了
        drag_coefficient = drag_coefficient,
        frontal_area_m2 = frontal_area_m2,
    ), axis=1)

    dc_df['fuel_rate'] = dc_df.apply(lambda row: lvd.calculate_diesel_consumption_rate(
        wheel_power_watts=row['p_wheel'],
        heating_value_mj_l= heating_value_mj_l,  # 燃料的低位热值 (MJ/L)
        engine_efficiency=engine_efficiency,   # 发动机效率
        powertrain_efficiency=powertrain_efficiency,  # 传动效率
        idle_fuel_consumption_l_per_hr=0.0  # 怠速燃油消耗率 (L/h)
    ), axis=1)

    # 计算燃油消耗量 (L)
    dc_df['fuel_use_cum'] = (dc_df['fuel_rate'] * (dc_df['timestamp'].diff().dt.total_seconds().fillna(0) / 3600)).cumsum()  # L

    # 通过FuelRate计算fuel_use_measured
    raw_df['fuel_use_measured'] = raw_df['FuelRate'] * (raw_df['timestamp'].diff().dt.total_seconds().fillna(0) / 3600)  # L
    raw_df['fuel_use_measured'] = raw_df['fuel_use_measured'].cumsum()  # L

    # 计算measurement speed profile的功率和燃油消耗率
    raw_df['p_wheel'] = raw_df.apply(lambda row: lvd.calculate_wheel_power(
        mass_kg=v_mass,
        gradient_degrees=row['road_gradient'],
        velocity_mps=row['Spd_Kmph_x'] / 3.6,  # 转换为 m/s
        acceleration_mps2=row['Acc_mps2'],
        rolling_resistance_coeff=rolling_resistance_coeff,
        drag_coefficient=drag_coefficient,
        frontal_area_m2=frontal_area_m2,
    ), axis=1)

    raw_df['fuel_rate'] = raw_df.apply(lambda row: lvd.calculate_diesel_consumption_rate(
        wheel_power_watts=row['p_wheel'],
        heating_value_mj_l= heating_value_mj_l,  # 燃料的低位热值 (MJ/L)
        engine_efficiency=engine_efficiency,   # 发动机效率
        powertrain_efficiency=powertrain_efficiency,  # 传动效率
        idle_fuel_consumption_l_per_hr=0.0  # 怠速燃油消耗率 (L/h)
    ), axis=1)
    # 计算燃油消耗量 (L)
    raw_df['fuel_use_cum'] = (raw_df['fuel_rate'] * (raw_df['timestamp'].diff().dt.total_seconds().fillna(0) / 3600)).cumsum()  # L

    # 用表格记录每50km的燃油消耗量以及模型误差

    max_distance = max(raw_df['distance_gps'])
    distance_intervals = np.arange(0, max_distance, 50000)
    if distance_intervals[-1] < max_distance:
        distance_intervals = np.append(distance_intervals, max_distance)

    distance_labels = [(min(end / 1000, max_distance / 1000)) for end in distance_intervals[1:]]

    fuel_use_stats = pd.DataFrame({
        'Distance (km)': distance_labels,
        'Measurement (L)': [
            raw_df[(raw_df['distance_gps'] >= start) & (raw_df['distance_gps'] < end)]['fuel_use_measured'].iloc[-1]
            if not raw_df[(raw_df['distance_gps'] >= start) & (raw_df['distance_gps'] < end)].empty else np.nan
            for start, end in zip(distance_intervals[:-1], distance_intervals[1:])
        ],
        'Simulation (L)': [
            raw_df[(raw_df['distance_gps'] >= start) & (raw_df['distance_gps'] < end)]['fuel_use_cum'].iloc[-1]
            if not raw_df[(raw_df['distance_gps'] >= start) & (raw_df['distance_gps'] < end)].empty else np.nan
            for start, end in zip(distance_intervals[:-1], distance_intervals[1:])
        ],
        'Pred. & Simulation (L)': [
            dc_df[(dc_df['distance'] >= start) & (dc_df['distance'] < end)]['fuel_use_cum'].iloc[-1]
            if not dc_df[(dc_df['distance'] >= start) & (dc_df['distance'] < end)].empty else np.nan
            for start, end in zip(distance_intervals[:-1], distance_intervals[1:])
        ],
    })

    fuel_use_stats['Simulation Error (%)'] = (fuel_use_stats['Simulation (L)'] - fuel_use_stats['Measurement (L)']) / fuel_use_stats['Measurement (L)'] * 100
    fuel_use_stats['Pred. & Simulation Error (%)'] = (fuel_use_stats['Pred. & Simulation (L)'] - fuel_use_stats['Measurement (L)']) / fuel_use_stats['Measurement (L)'] * 100
    fuel_use_stats['Avg Wind Speed (m/s)'] = avg_wind_speed
    fuel_use_stats = fuel_use_stats.round(2)
    # 保存 fuel_use_stats
    fuel_use_stats.to_csv(os.path.join(result_save_dir, 'fuel_use_stats.csv'), index=False)

    # 画一个总的report图，包括5个子图，5*1的网格布局
    report_fig, report_axes = plt.subplots(5, 1, figsize=(12, 15), sharex=False)

    # 添加总标题
    report_fig.suptitle('Driving Cycle Report for ' + file_name, fontsize=18, y=1.02)

    # 子图1：速度随时间变化
    ax1 = report_axes[0]
    ax1.plot(raw_df["timestamp"], raw_df["Spd_Kmph_x"], label='Measurement', color='red', linewidth=1)
    ax1.plot(dc_df['timestamp'], dc_df['speed']*3.6, label='Prediction', color=color_dict['simulation+pred'], linewidth=1)
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax1.set_xlabel('Time', fontsize=14)
    ax1.set_ylabel('Speed (km/h)', fontsize=14)
    ax1.set_ylim(0, 100)
    ax1.legend(fontsize=12, loc='best')
    ax1.grid()
    ax1.tick_params(axis='x', labelsize=12)
    ax1.tick_params(axis='y', labelsize=12)
    ax1.set_title('Driving Cycle Speed vs Time', fontsize=15)
    for label in ax1.get_xticklabels():
        label.set_rotation(45)

    # 统一distance相关x轴刻度，范围从0到xmax
    xmax = max(raw_df["distance_gps"]) / 1000
    xlim = (0, xmax)
    if xmax < 50:
        xticks = list(np.arange(0, xmax, 10))
    else:
        xticks = list(np.arange(0, xmax, 10))
    if not np.isclose(xmax, xticks[-1]):
        xticks.append(round(xmax, 2))

    # 子图2：速度随距离变化
    ax2 = report_axes[1]
    ax2.plot(raw_df["distance_gps"]/1000, raw_df["Spd_Kmph_x"], label='Measurement', color='red', linewidth=1)
    ax2.plot(dc_df['distance']/1000, dc_df['speed']*3.6, label='Prediction', color=color_dict['simulation+pred'], linewidth=1)
    ax2.set_xlabel('Distance (km)', fontsize=14)
    ax2.set_ylabel('Speed (km/h)', fontsize=14)
    ax2.set_xlim(xlim)
    ax2.set_ylim(0, 100)
    ax2.set_xticks(xticks)
    ax2.legend(fontsize=12, loc='best')
    ax2.grid()
    ax2.tick_params(axis='x', labelsize=12)
    ax2.tick_params(axis='y', labelsize=12)
    ax2.set_title('Driving Cycle Speed vs Distance', fontsize=15)
    for label in ax2.get_xticklabels():
        label.set_rotation(45)

    # 子图3：车辆质量随距离变化
    ax3 = report_axes[2]
    ax3.plot(raw_df['distance_gps']/1000, raw_df['MassKg'], label='Measurement', color='red')
    ax3.axhline(y=v_mass, color=color_dict['simulation'], linestyle='--', label=f'Average Mass: {v_mass:.2f} kg')
    ax3.set_xlabel('Distance (km)', fontsize=14)
    ax3.set_ylabel('Mass (kg)', fontsize=14)
    ax3.set_xlim(xlim)
    ax3.set_xticks(xticks)
    ax3.tick_params(axis='x', labelsize=12)
    ax3.tick_params(axis='y', labelsize=12)
    ax3.grid()
    ax3.legend(fontsize=12, loc='upper left')
    ax3.set_title('Vehicle Mass Profile by Distance', fontsize=15)
    for label in ax3.get_xticklabels():
        label.set_rotation(45)

    # 子图4：燃油消耗量关于距离的对比曲线
    ax4 = report_axes[3]
    ax4.plot(raw_df['distance_gps']/1000, raw_df['fuel_use_measured'], label='Measurement', color='red', linewidth=2)
    ax4.plot(raw_df['distance_gps']/1000, raw_df['fuel_use_cum'], label='Simulation', color=color_dict['simulation'], linewidth=2)
    ax4.plot(dc_df['distance']/1000, dc_df['fuel_use_cum'], label='Pred. & Simulation', color=color_dict['simulation+pred'], linewidth=2)
    ax4.set_xlabel('Distance (km)', fontsize=14)
    ax4.set_ylabel('Fuel Use (L)', fontsize=14)
    ax4.set_xlim(xlim)
    ax4.set_xticks(xticks)
    ax4.legend(fontsize=12, loc='lower right')
    ax4.grid()
    ax4.tick_params(axis='x', labelsize=12)
    ax4.tick_params(axis='y', labelsize=12)
    ax4.set_title('Fuel Consumption vs. Distance', fontsize=15)
    for label in ax4.get_xticklabels():
        label.set_rotation(45)

    # 子图5：每50km燃油消耗统计
    ax5 = report_axes[4]
    # 在数据最前面增加一个(0, 0)的点
    distances = [0.0] + fuel_use_stats["Distance (km)"].tolist()
    model_errors = [0.0] + fuel_use_stats["Simulation Error (%)"].tolist()
    pred_errors = [0.0] + fuel_use_stats["Pred. & Simulation Error (%)"].tolist()

    ax5.plot(distances, model_errors, marker="s", label="Simulation Error (%)", color=color_dict['simulation'], linewidth=2)
    ax5.plot(distances, pred_errors, marker="^", label="Pred. & Simulation Error (%)", color=color_dict['simulation+pred'], linewidth=2)

    # 标注百分比误差数值，确保标注不会画到图外
    ymin, ymax = ax5.get_ylim()
    offset_above = 8
    offset_below = -15
    margin = 2  # 距离y轴上下边界的最小距离

    for x, y in zip(distances, model_errors):
        # 限制标注y值在ymin+margin和ymax-margin之间
        if y >= 0:
            y_annot = min(y, ymax - margin)
            xytext = (0, offset_above)
            va = 'bottom'
        else:
            y_annot = max(y, ymin + margin)
            xytext = (0, offset_below)
            va = 'top'
        ax5.annotate(
            f"{y:.1f}%", 
            (x, y_annot), 
            textcoords="offset points", 
            xytext=xytext, 
            ha='center', 
            va=va,
            fontsize=10, 
            color=color_dict['simulation'],
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.5)
        )

    for x, y in zip(distances, pred_errors):
        if y >= 0:
            y_annot = min(y, ymax - margin)
            xytext = (0, offset_above)
            va = 'bottom'
        else:
            y_annot = max(y, ymin + margin)
            xytext = (0, offset_below)
            va = 'top'
        ax5.annotate(
            f"{y:.1f}%", 
            (x, y_annot), 
            textcoords="offset points", 
            xytext=xytext, 
            ha='center', 
            va=va,
            fontsize=10, 
            color=color_dict['simulation+pred'],
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.5)
        )

    ax5.tick_params(axis='both', labelsize=12)
    ax5.set_xticks(xticks)
    ax5.grid(True, linestyle='--', alpha=0.5)
    ax5.legend(fontsize=12, loc='best')
    ax5.set_xlabel("Distance (km)", fontsize=14)
    ax5.set_ylabel("Percentage Error (%)", fontsize=14)
    ax5.set_title("Fuel Consumption Percentage Error (per 50km)", fontsize=15)
    for label in ax5.get_xticklabels():
        label.set_rotation(45)

    plt.tight_layout()
    report_fig.savefig(os.path.join(result_save_dir, "driving_cycle_report.png"), bbox_inches='tight', dpi=300)
    plt.show()
    